## P3: Funciones compartidas

- Parámetros y funciones de TF-IDF, producto punto disperso nativo, índice IVF (KMeans) y top-k
- No se ejecuta solo: los notebooks a, b, c y d lo cargan con `%run`

In [ ]:
import numpy as np
from pyspark.sql import Window, functions as F, types as T
from pyspark.ml.feature import HashingTF, IDF, Normalizer
from pyspark.ml.clustering import KMeans
from pyspark.ml.linalg import SparseVector, DenseVector

# ---- parámetros ----
SEED = 42
NUM_FEATURES = 4096        # dimensiones del hashing TF-IDF
MIN_DF = 2                 # un token debe aparecer en >= 2 procesos para tener IDF
NLIST = 300                # nº de listas invertidas (clusters) del IVF
NPROBE = 2                 # listas que se revisan por consulta
K = 10                     # vecinos por consulta
N_QUERIES = 20_000         # consultas en la corrida ANN (muestra)
N_EXACT = 300              # consultas para comparar con fuerza bruta
UMBRAL_SIM = 0.80          # coseno mínimo para considerar "muy parecidos" en el análisis
ANIOS = [2023, 2024, 2025]

COLS_P3 = ["ocid", "anio", "mes", "periodo", "entidad", "region", "categoria", "metodo_detalle",
           "proveedor", "monto_final", "tramo_monto", "descripcion", "tokens"]


# ---- TF-IDF (lab3.2: TfidfVectorizer + coseno) ----
def ajustar_tfidf(df, col_tokens="tokens", num_features=NUM_FEATURES, min_df=MIN_DF):
    # HashingTF (sin vocabulario en memoria) + IDF + normalización L2 => coseno = producto punto
    htf = HashingTF(inputCol=col_tokens, outputCol="tf", numFeatures=num_features)
    tf = htf.transform(df)
    idf = IDF(inputCol="tf", outputCol="tfidf", minDocFreq=min_df).fit(tf)
    norm = Normalizer(inputCol="tfidf", outputCol="vec", p=2.0)
    return norm.transform(idf.transform(tf)).drop("tf", "tfidf"), idf


@F.udf(T.StructType([T.StructField("idx", T.ArrayType(T.IntegerType())), T.StructField("val", T.ArrayType(T.DoubleType()))]))
def vector_a_arrays(v):
    # vector disperso -> (índices, valores) para poder usar funciones nativas de Spark
    if isinstance(v, DenseVector):
        v = SparseVector(len(v), {i: x for i, x in enumerate(v) if x != 0})
    return ([int(i) for i in v.indices], [float(x) for x in v.values])


def coseno(idx_a, val_a, idx_b, val_b):
    # producto punto disperso nativo: suma de val_a[k]*val_b[k] sobre los índices comunes (vectores ya normalizados)
    ma, mb = F.map_from_arrays(idx_a, val_a), F.map_from_arrays(idx_b, val_b)
    return F.aggregate(F.array_intersect(idx_a, idx_b), F.lit(0.0), lambda acc, k: acc + F.element_at(ma, k) * F.element_at(mb, k))


# ---- IVF: cuantizador grueso (KMeans) + listas invertidas ----
def entrenar_centroides(df_vec, nlist=NLIST, seed=SEED, max_iter=10):
    km = KMeans(k=nlist, seed=seed, maxIter=max_iter, featuresCol="vec", predictionCol="lista").fit(df_vec)
    return km, np.vstack([np.asarray(c) for c in km.clusterCenters()])


def asignar_listas_udf(centroides_bc, nprobe=NPROBE):
    # para cada vector: las nprobe listas más cercanas (la 1ª es donde se indexa; todas se usan al consultar)
    norm_c = np.array([float(np.dot(c, c)) for c in centroides_bc.value])

    @F.udf(T.ArrayType(T.IntegerType()))
    def f(idx, val):
        C = centroides_bc.value
        dots = C[:, np.array(idx, dtype=int)] @ np.array(val)
        dist = norm_c - 2 * dots                 # ||x - c||² sin el término ||x||² (constante)
        return [int(i) for i in np.argsort(dist)[:nprobe]]
    return f


def top_k(df, col_q="ocid_q", col_sim="cos_sim", k=K):
    w = Window.partitionBy(col_q).orderBy(F.desc(col_sim), "ocid_v")
    return df.withColumn("rank", F.row_number().over(w)).where(F.col("rank") <= k)


def renombrar(df, sufijo, excluir=()):
    return df.select(*[F.col(c).alias(f"{c}_{sufijo}") if c not in excluir else F.col(c) for c in df.columns])